# 🧮 Module 02: Neural Network Fundamentals from First Principles
### Pure NumPy Multi-Layer Perceptron (MLP) & Activation Function Benchmark

This notebook implements:
1. Core mathematical activation functions and their derivatives: **Sigmoid, ReLU, Tanh, and Softmax**.
2. Multi-class Cross-Entropy Loss and its analytical derivative w.r.t pre-activation logits.
3. 2-layer Multi-Layer Perceptron built from scratch with matrix operations in pure NumPy.
4. Momentum-based Stochastic Gradient Descent (SGD) with L2 weight decay.
5. Experimental comparison of convergence dynamics: **ReLU vs. Sigmoid vs. Tanh**.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams['figure.figsize'] = (10, 6)
sns.set_theme(style="whitegrid")
np.random.seed(42)

### 2.1 Activation Functions & Analytical Derivatives

In [ ]:
class Activation:
    @staticmethod
    def sigmoid(x):
        clipped = np.clip(x, -500, 500)
        return 1.0 / (1.0 + np.exp(-clipped))

    @staticmethod
    def sigmoid_derivative(x):
        s = Activation.sigmoid(x)
        return s * (1.0 - s)

    @staticmethod
    def relu(x):
        return np.maximum(0, x)

    @staticmethod
    def relu_derivative(x):
        return (x > 0).astype(float)

    @staticmethod
    def tanh(x):
        return np.tanh(x)

    @staticmethod
    def tanh_derivative(x):
        return 1.0 - np.tanh(x) ** 2

    @staticmethod
    def softmax(x):
        shift_x = x - np.max(x, axis=1, keepdims=True)
        exps = np.exp(shift_x)
        return exps / np.sum(exps, axis=1, keepdims=True)

# Visualizing Activation Curves and Derivatives
x_vals = np.linspace(-5, 5, 200)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4.5))

ax1.plot(x_vals, Activation.sigmoid(x_vals), label="Sigmoid", color="#1f77b4", linewidth=2)
ax1.plot(x_vals, Activation.relu(x_vals), label="ReLU", color="#2ca02c", linewidth=2)
ax1.plot(x_vals, Activation.tanh(x_vals), label="Tanh", color="#ff7f0e", linewidth=2)
ax1.set_title("Activation Functions $f(x)$", fontsize=12, fontweight="bold")
ax1.set_ylim(-1.5, 5)
ax1.legend()
ax1.grid(True, linestyle=":", alpha=0.6)

ax2.plot(x_vals, Activation.sigmoid_derivative(x_vals), label="Sigmoid'", color="#1f77b4", linewidth=2)
ax2.plot(x_vals, Activation.relu_derivative(x_vals), label="ReLU'", color="#2ca02c", linewidth=2)
ax2.plot(x_vals, Activation.tanh_derivative(x_vals), label="Tanh'", color="#ff7f0e", linewidth=2)
ax2.set_title("Analytical Derivatives $f'(x)$", fontsize=12, fontweight="bold")
ax2.set_ylim(-0.2, 1.2)
ax2.legend()
ax2.grid(True, linestyle=":", alpha=0.6)

plt.tight_layout()
plt.show()

### 2.2 Cross-Entropy Loss & Softmax Gradient

In [ ]:
class Loss:
    @staticmethod
    def cross_entropy(y_pred, y_true):
        eps = 1e-12
        n_samples = y_pred.shape[0]
        p = np.clip(y_pred[np.arange(n_samples), y_true], eps, 1.0 - eps)
        return -np.mean(np.log(p))

    @staticmethod
    def cross_entropy_grad(y_pred, y_true):
        n_samples = y_pred.shape[0]
        grad = y_pred.copy()
        grad[np.arange(n_samples), y_true] -= 1.0
        return grad / n_samples

### 2.3 Pure NumPy 2-Layer MLP Architecture

In [ ]:
class NumPyMLP:
    """2-Layer Multi-Layer Perceptron built purely from scratch using NumPy."""
    def __init__(self, input_dim, hidden_dim, output_dim, activation="relu", seed=42):
        np.random.seed(seed)
        self.activation_name = activation.lower()
        if self.activation_name == "relu":
            scale1 = np.sqrt(2.0 / input_dim)
            scale2 = np.sqrt(2.0 / hidden_dim)
        else:
            scale1 = np.sqrt(1.0 / input_dim)
            scale2 = np.sqrt(1.0 / hidden_dim)

        self.W1 = np.random.randn(input_dim, hidden_dim) * scale1
        self.b1 = np.zeros((1, hidden_dim))
        self.W2 = np.random.randn(hidden_dim, output_dim) * scale2
        self.b2 = np.zeros((1, output_dim))

        self.v_W1 = np.zeros_like(self.W1)
        self.v_b1 = np.zeros_like(self.b1)
        self.v_W2 = np.zeros_like(self.W2)
        self.v_b2 = np.zeros_like(self.b2)

    def _activate(self, z):
        if self.activation_name == "relu": return Activation.relu(z)
        elif self.activation_name == "sigmoid": return Activation.sigmoid(z)
        elif self.activation_name == "tanh": return Activation.tanh(z)
        raise ValueError(f"Unknown activation: {self.activation_name}")

    def _activate_derivative(self, z):
        if self.activation_name == "relu": return Activation.relu_derivative(z)
        elif self.activation_name == "sigmoid": return Activation.sigmoid_derivative(z)
        elif self.activation_name == "tanh": return Activation.tanh_derivative(z)
        raise ValueError(f"Unknown activation: {self.activation_name}")

    def forward(self, X):
        self.X = X
        self.Z1 = np.dot(X, self.W1) + self.b1
        self.A1 = self._activate(self.Z1)
        self.Z2 = np.dot(self.A1, self.W2) + self.b2
        self.probs = Activation.softmax(self.Z2)
        return self.probs

    def backward(self, y_true):
        dZ2 = Loss.cross_entropy_grad(self.probs, y_true)
        self.dW2 = np.dot(self.A1.T, dZ2)
        self.db2 = np.sum(dZ2, axis=0, keepdims=True)

        dA1 = np.dot(dZ2, self.W2.T)
        dZ1 = dA1 * self._activate_derivative(self.Z1)
        self.dW1 = np.dot(self.X.T, dZ1)
        self.db1 = np.sum(dZ1, axis=0, keepdims=True)

    def step(self, lr=0.01, momentum=0.9, l2_reg=1e-4):
        self.dW2 += l2_reg * self.W2
        self.dW1 += l2_reg * self.W1

        self.v_W2 = momentum * self.v_W2 - lr * self.dW2
        self.v_b2 = momentum * self.v_b2 - lr * self.db2
        self.v_W1 = momentum * self.v_W1 - lr * self.dW1
        self.v_b1 = momentum * self.v_b1 - lr * self.db1

        self.W2 += self.v_W2
        self.b2 += self.v_b2
        self.W1 += self.v_W1
        self.b1 += self.v_b1

    def train_epoch(self, X, y, lr=0.01, momentum=0.9, l2_reg=1e-4):
        probs = self.forward(X)
        loss = Loss.cross_entropy(probs, y)
        self.backward(y)
        self.step(lr=lr, momentum=momentum, l2_reg=l2_reg)
        preds = np.argmax(probs, axis=1)
        acc = np.mean(preds == y)
        return loss, acc

### 2.4 Empirical Benchmark: ReLU vs. Sigmoid vs. Tanh

In [ ]:
np.random.seed(42)
n_samples, n_features, n_classes = 400, 32, 4
X_syn = np.random.randn(n_samples, n_features)
y_syn = (np.sum(X_syn[:, :4] ** 2, axis=1) + np.sin(X_syn[:, 4]) * 2).astype(int) % n_classes

activation_results = {}
epochs = 60
for act in ["relu", "sigmoid", "tanh"]:
    mlp = NumPyMLP(input_dim=n_features, hidden_dim=64, output_dim=n_classes, activation=act)
    hist = []
    for ep in range(epochs):
        loss_val, acc_val = mlp.train_epoch(X_syn, y_syn, lr=0.05, momentum=0.9)
        hist.append((loss_val, acc_val))
    activation_results[act] = hist

# Comparative Plots
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
color_map = {"relu": "#2ca02c", "sigmoid": "#1f77b4", "tanh": "#ff7f0e"}

for act, hist in activation_results.items():
    losses = [h[0] for h in hist]
    accs = [h[1] * 100 for h in hist]
    ax1.plot(range(1, epochs + 1), losses, label=act.upper(), color=color_map[act], linewidth=2.2)
    ax2.plot(range(1, epochs + 1), accs, label=act.upper(), color=color_map[act], linewidth=2.2)

ax1.set_title("Training Loss Convergence by Activation", fontsize=12, fontweight="bold")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Cross Entropy Loss")
ax1.legend()
ax1.grid(True, linestyle=":", alpha=0.6)

ax2.set_title("Training Accuracy Convergence by Activation", fontsize=12, fontweight="bold")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Accuracy (%)")
ax2.legend()
ax2.grid(True, linestyle=":", alpha=0.6)

plt.tight_layout()
plt.show()

print("Convergence Benchmark Summary:")
for act, hist in activation_results.items():
    print(f"  Activation: {act.upper():<8} -> Final Loss: {hist[-1][0]:.4f} | Final Accuracy: {hist[-1][1]*100:.1f}%")